# Domain Classification Experiments

Load extracted domain features from `domain_analysis/extractor/output`, build a labeled dataset, and evaluate multiple classifiers.

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

# Locate project root (folder that contains README.md)
project_root = Path.cwd()
for _ in range(6):
    if (project_root / "README.md").exists():
        break
    project_root = project_root.parent

data_dir = project_root / "domain_analysis" / "extractor" / "output"
benign_path = data_dir / "benign_umbrella_features.csv"
phish_path = data_dir / "phishing_features.csv"

if not benign_path.exists() or not phish_path.exists():
    raise FileNotFoundError(f"Dataset files not found in {data_dir}")

df_benign = pd.read_csv(benign_path)
df_phish = pd.read_csv(phish_path)

print("benign shape:", df_benign.shape)
print("phish shape:", df_phish.shape)

# Align columns if needed
common_cols = df_benign.columns.intersection(df_phish.columns)
if len(common_cols) != len(df_benign.columns) or len(common_cols) != len(df_phish.columns):
    missing_benign = sorted(set(df_phish.columns) - set(df_benign.columns))
    missing_phish = sorted(set(df_benign.columns) - set(df_phish.columns))
    print("Columns missing in benign:", missing_benign[:10])
    print("Columns missing in phish:", missing_phish[:10])

df_benign = df_benign[common_cols]
df_phish = df_phish[common_cols]

df = pd.concat([df_benign, df_phish], ignore_index=True)
print("merged shape:", df.shape)
display(df.head(3))

benign shape: (2000, 139)
phish shape: (2000, 139)
merged shape: (4000, 139)


,domain_name,dns_has_dnskey,dns_dnssec_score,dns_zone_level,dns_zone_digit_count,dns_zone_len,dns_zone_entropy,dns_resolved_record_types,dns_ttl_avg,dns_ttl_stdev,...,rdap_ip_v6_count,rdap_ip_shortest_v4_prefix_len,rdap_ip_longest_v4_prefix_len,rdap_ip_shortest_v6_prefix_len,rdap_ip_longest_v6_prefix_len,rdap_ip_avg_admin_name_len,rdap_ip_avg_admin_name_entropy,rdap_ip_avg_admin_email_len,rdap_ip_avg_admin_email_entropy,class
0,hme-live-nitro-feedback-service.hmecloud.com,0,0.0,0,0,12,0.257080,1,450.0,1190.588090,...,0,14,14,0,0,0.0,0.000000,0.0,0.000000,benign
1,www.4digitalsignage.com,0,0.0,0,1,19,0.191692,1,450.0,1190.588090,...,0,11,11,0,0,13.0,0.249146,23.0,0.149881,benign
2,www.douyin.com.bytedns1.com,0,0.0,0,1,12,0.298747,1,7.5,19.843135,...,0,18,18,0,0,0.0,0.000000,0.0,0.000000,benign


In [2]:
# Prepare features and labels
df = df.copy()
df["class"] = df["class"].astype(str).str.lower()

label_map = {
    "benign": 0,
    "phish": 1,
    "phishing": 1,
    "1": 1,
    "0": 0,
    "true": 1,
    "false": 0,
}
y = df["class"].map(label_map)

if y.isna().any():
    y = pd.to_numeric(df["class"], errors="coerce")

valid_mask = y.isin([0, 1])
df = df.loc[valid_mask].copy()
y = y.loc[valid_mask].astype(int)

drop_cols = ["class"]
if "domain_name" in df.columns:
    drop_cols.append("domain_name")

X = df.drop(columns=drop_cols)
X = X.replace({True: 1, False: 0})
X = X.apply(pd.to_numeric, errors="coerce")

print("X shape:", X.shape)
print("phish rate:", float(y.mean()))
missing_rate = X.isna().mean().sort_values(ascending=False)
display(missing_rate.head(10))

X shape: (4000, 137)
phish rate: 0.5


dns_mx_avg_entropy                0.84750
dns_mx_avg_len                    0.84750
dns_txt_avg_entropy               0.82025
dns_txt_avg_len                   0.80325
dns_soa_primary_ns_level          0.67200
dns_soa_primary_ns_digit_count    0.67200
dns_soa_primary_ns_len            0.67200
dns_soa_primary_ns_entropy        0.67200
dns_soa_retry                     0.67200
dns_soa_email_level               0.67200
dtype: float64

In [3]:
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, HistGradientBoostingClassifier
from sklearn.svm import LinearSVC
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score,
)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

def make_pipeline(model, scale=True):
    steps = [("imputer", SimpleImputer(strategy="median"))]
    if scale:
        steps.append(("scaler", StandardScaler()))
    steps.append(("model", model))
    return Pipeline(steps)

models = {
    "LogisticRegression": make_pipeline(
        LogisticRegression(max_iter=500, class_weight="balanced")
    ),
    "RandomForest": make_pipeline(
        RandomForestClassifier(
            n_estimators=300,
            random_state=42,
            class_weight="balanced",
            n_jobs=-1,
        ),
        scale=False,
    ),
    "GradientBoosting": make_pipeline(
        GradientBoostingClassifier(random_state=42),
        scale=False,
    ),
    "HistGradientBoosting": make_pipeline(
        HistGradientBoostingClassifier(random_state=42),
        scale=False,
    ),
    "LinearSVC": make_pipeline(
        LinearSVC(class_weight="balanced", random_state=42)
    ),
}

def evaluate_model(name, model, X_train, X_test, y_train, y_test):
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    metrics = {
        "model": name,
        "accuracy": accuracy_score(y_test, y_pred),
        "precision": precision_score(y_test, y_pred, zero_division=0),
        "recall": recall_score(y_test, y_pred, zero_division=0),
        "f1": f1_score(y_test, y_pred, zero_division=0),
    }

    y_score = None
    if hasattr(model, "predict_proba"):
        y_score = model.predict_proba(X_test)[:, 1]
    elif hasattr(model, "decision_function"):
        y_score = model.decision_function(X_test)

    if y_score is not None:
        metrics["roc_auc"] = roc_auc_score(y_test, y_score)
        metrics["avg_precision"] = average_precision_score(y_test, y_score)
    else:
        metrics["roc_auc"] = np.nan
        metrics["avg_precision"] = np.nan

    return metrics, y_pred

results = []
predictions = {}

for name, model in models.items():
    metrics, y_pred = evaluate_model(name, model, X_train, X_test, y_train, y_test)
    results.append(metrics)
    predictions[name] = y_pred

results_df = (
    pd.DataFrame(results)
    .set_index("model")
    .sort_values(["f1", "roc_auc"], ascending=False)
)

display(results_df)

,accuracy,precision,recall,f1,roc_auc,avg_precision
model,,,,,,
HistGradientBoosting,0.97125,0.994751,0.9475,0.970551,0.996356,0.996730
GradientBoosting,0.96625,0.984416,0.9475,0.965605,0.995169,0.995804
RandomForest,0.96125,0.989390,0.9325,0.960103,0.996094,0.996299
LogisticRegression,0.92875,0.952507,0.9025,0.926829,0.977000,0.980284
LinearSVC,0.92625,0.947507,0.9025,0.924456,0.976625,0.977981


In [4]:
from sklearn.metrics import confusion_matrix, classification_report

best_model_name = results_df.index[0]
best_model = models[best_model_name]

best_model.fit(X_train, y_train)
best_pred = best_model.predict(X_test)

print("Best model:", best_model_name)
print(classification_report(y_test, best_pred, target_names=["benign", "phish"]))

cm = confusion_matrix(y_test, best_pred)
cm_df = pd.DataFrame(
    cm,
    index=["true_benign", "true_phish"],
    columns=["pred_benign", "pred_phish"],
)

display(cm_df)

Best model: HistGradientBoosting
              precision    recall  f1-score   support

      benign       0.95      0.99      0.97       400
       phish       0.99      0.95      0.97       400

    accuracy                           0.97       800
   macro avg       0.97      0.97      0.97       800
weighted avg       0.97      0.97      0.97       800



,pred_benign,pred_phish
true_benign,398,2
true_phish,21,379
